In [2]:
from google.colab import userdata
secret_value_0 = userdata.get('HF_TOKEN')

In [3]:
from datasets import load_dataset
shards = ["data/000_00000.parquet"]
dataset_name = "salyamq/kk-corpus-v1"


ds = load_dataset(dataset_name, split="train", verification_mode="no_checks",  data_files={"train": shards}, token = secret_value_0)

data/000_00000.parquet: reconstructing file:   0%|          |  0.00B /  110MB            

data/000_00000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
ds["text"][999]

'ШЫМКЕНТ ҚАЛАСЫНЫҢ ӘКІМІ АЛҒАШҚЫ ЖИНАЛЫСЫН ӨТКІЗДІ\nШымкент қаласының әкімі Ерлан Айтаханов 31 шілде күні өз орынбасарларын, аудан әкімдерін, салалық басқарма басшыларын жинап, алғашқы жиналысын өткізді. Мегаполис әкімі мәжілістің басында аудандардың және басқармалардың жұмыс бағыттарымен танысты.\nЕрлан Қуанышұлы ең өзекті мәселелердің бірі ретінде қаланың шалғай елдімекендерін дамыту қажеттігін басып айтты. Қажет инфрақұрылымның, атап айтқанда, сапалы ауызсу, газбен жабдықтау, электр қуаты, жол, кәріз жүйелерінің жеткіліксіздігі мәселесін жүйелі түрде шешуді талап етті. Келесі жылдың басымды бағыты ретінде қаланың шалғай елдімекендерін дамытуды айқындады.\n– Жергілікті билік, қаланың әр тұрғынына, олардың тұрғылықты мекенжайына қарамастан бірдей жағдай жасауы қажет, яғни, өмір сүруге қолайлы әрі қауіпсіз орта қалыптастыру, бұл – Елбасымыз және Президентіміздің әкімдерге қойған басты талабы, - деп ашып айтты Ерлан Қуанышұлы.\nӘкім, сонымен қатар, тұрғындардың мәселелерін жедел әрі сап

In [6]:
import random

random.seed(511)

texts = ds["text"]
n = len(texts)
print(f"total doc: {n}")

sample_size = 20000
indices = random.sample(range(n), min(sample_size, n))

with open("sample_corpus.txt", "w", encoding="utf-8") as f:
    for i in indices:
        line = texts[i].strip()
        if line:
            f.write(line + "\n")

print(f"subset: {len(indices)} docs")

total doc: 44218
subset: 20000 docs


In [7]:
import sentencepiece as spm
from tqdm import tqdm
import os

corpus_path = "sample_corpus.txt"
vocab_sizes = [8000, 16000, 32000, 50000, 64000]
algorithms = ["unigram", "bpe"]


os.makedirs("tokenizers", exist_ok=True)

for alg in tqdm(algorithms, desc="algorithms"):
    for vs in vocab_sizes:
        model_prefix = f"tokenizers/{alg}_{vs}"

        print(f"training: {model_prefix} ...")
        spm.SentencePieceTrainer.train(
            input=corpus_path,
            model_prefix=model_prefix,
            vocab_size=vs,
            model_type=alg,
            character_coverage=0.9995,
            input_sentence_size=1000000,
            pad_id=0, unk_id=1, bos_id=2, eos_id=3,
            hard_vocab_limit=False,
            byte_fallback=False,
            shuffle_input_sentence=True
        )
        print(f"done {model_prefix}.model / {model_prefix}.vocab\n")

algorithms:   0%|          | 0/2 [00:00<?, ?it/s]

training: tokenizers/unigram_8000 ...
done tokenizers/unigram_8000.model / tokenizers/unigram_8000.vocab

training: tokenizers/unigram_16000 ...
done tokenizers/unigram_16000.model / tokenizers/unigram_16000.vocab

training: tokenizers/unigram_32000 ...
done tokenizers/unigram_32000.model / tokenizers/unigram_32000.vocab

training: tokenizers/unigram_50000 ...
done tokenizers/unigram_50000.model / tokenizers/unigram_50000.vocab

training: tokenizers/unigram_64000 ...


algorithms:  50%|█████     | 1/2 [28:23<28:23, 1703.81s/it]

done tokenizers/unigram_64000.model / tokenizers/unigram_64000.vocab

training: tokenizers/bpe_8000 ...
done tokenizers/bpe_8000.model / tokenizers/bpe_8000.vocab

training: tokenizers/bpe_16000 ...
done tokenizers/bpe_16000.model / tokenizers/bpe_16000.vocab

training: tokenizers/bpe_32000 ...
done tokenizers/bpe_32000.model / tokenizers/bpe_32000.vocab

training: tokenizers/bpe_50000 ...
done tokenizers/bpe_50000.model / tokenizers/bpe_50000.vocab

training: tokenizers/bpe_64000 ...


algorithms: 100%|██████████| 2/2 [29:49<00:00, 894.61s/it]

done tokenizers/bpe_64000.model / tokenizers/bpe_64000.vocab



In [9]:
import sentencepiece as spm
import json
import os
import glob

corpus_path = "chunk_099.txt"

test_tokenizer_dict = {
    "балаларға": "бала+лар+ға",
    "келгендеріңіз": "кел+ген+дер+іңіз",
    "жаздыртылмаған": "жаз+дыр+т+ыл+ма+ған",
    "оқушыларымыздың": "оқу+шы+лар+ымыз+дың",
    "сөйлесетінмін": "сөйле+с+е+тін+мін",
    "тапсырылды": "тапсыр+ыл+ды",
    "бармайтынбыз": "бар+ма+йтын+быз",
    "сұлулығымен": "сұлу+лық+ы+мен",
    "адамдарша": "адам+дар+ша",
    "жұмысшылармен": "жұмыс+шы+лар+мен",
    "бүгінгі": "бүгін+гі",
    "теміржолшы": "темір+жол+шы",
    "ақпараттандыру": "ақпарат+тан+дыр+у",
    "мемлекеттік": "мемлекет+тік",
    "қолданбалы": "қолдан+ба+лы",
    "кітаптарым": "кітап+тар+ым",
    "мектепке": "мектеп+ке",
    "ағаштардың": "ағаш+тар+дың",
    "балалардан": "бала+лар+дан",
    "келмейді": "кел+ме+й+ді",
    "барғандар": "бар+ған+дар",
    "алмадым": "ал+ма+ды+м",
    "көрдік": "көр+ді+к",
    "жазамын": "жаз+а+мын",
    "кетеміз": "кет+е+міз",
    "оқулықтарыңыз": "оқу+лық+тар+ыңыз",
    "Атырау": "Атырау",
    "Тараз": "Тараз",
    "мен": "мен",
    "ие": "ие",
}


def load_corpus(path):
    with open(path, encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]


def corpus_metrics(sp, lines):
    total_words = total_tokens = total_chars = unk_count = 0
    token_lengths = []
    unique_tokens_used = set()

    for line in lines:
        words = line.split()
        total_words += len(words)
        total_chars += len(line.replace(" ", ""))

        tokens = sp.encode(line, out_type=str)
        total_tokens += len(tokens)
        unique_tokens_used.update(tokens)

        ids = sp.encode(line, out_type=int)
        unk_count += sum(1 for i in ids if i == sp.unk_id())
        token_lengths.extend(len(t.replace("▁", "")) for t in tokens)

    return {
        "total_words": total_words,
        "total_tokens": total_tokens,
        "total_chars": total_chars,
        "fertility_tokens_per_word": round(total_tokens / total_words, 4) if total_words else None,
        "avg_token_length_chars": round(sum(token_lengths) / len(token_lengths), 4) if token_lengths else None,
        "compression_chars_per_token": round(total_chars / total_tokens, 4) if total_tokens else None,
        "unk_rate": round(unk_count / total_tokens, 6) if total_tokens else None,
        "unique_tokens_used": len(unique_tokens_used),
        "vocab_size": sp.vocab_size(),
        "vocab_utilization": round(len(unique_tokens_used) / sp.vocab_size(), 4),
    }


def get_boundaries(piece_lengths):
    bounds, acc = set(), 0
    for l in piece_lengths[:-1]:
        acc += l
        bounds.add(acc)
    return bounds


def morpheme_metrics(sp, test_dict):
    exact_matches = total = 0
    precisions, recalls, f1s = [], [], []
    per_word = {}

    for word, gold in test_dict.items():
        gold_pieces = gold.split("+")
        if "".join(gold_pieces) != word:
            per_word[word] = {"skipped": True, "reason": "reconstruction_mismatch"}
            continue

        total += 1
        gold_bounds = get_boundaries([len(p) for p in gold_pieces])

        pred_pieces = [t.replace("▁", "") for t in sp.encode(word, out_type=str)]
        pred_pieces = [p for p in pred_pieces if p]
        pred_bounds = get_boundaries([len(p) for p in pred_pieces])

        exact = pred_bounds == gold_bounds
        exact_matches += exact

        tp = len(gold_bounds & pred_bounds)
        fp = len(pred_bounds - gold_bounds)
        fn = len(gold_bounds - pred_bounds)
        precision = tp / (tp + fp) if (tp + fp) else 1.0
        recall = tp / (tp + fn) if (tp + fn) else 1.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        precisions.append(precision); recalls.append(recall); f1s.append(f1)
        per_word[word] = {
            "gold": gold,
            "predicted": "+".join(pred_pieces),
            "exact_match": exact,
            "precision": round(precision, 3),
            "recall": round(recall, 3),
            "f1": round(f1, 3),
        }

    return {
        "morpheme_exact_match_accuracy": round(exact_matches / total, 4) if total else None,
        "morpheme_boundary_precision_avg": round(sum(precisions) / len(precisions), 4) if precisions else None,
        "morpheme_boundary_recall_avg": round(sum(recalls) / len(recalls), 4) if recalls else None,
        "morpheme_boundary_f1_avg": round(sum(f1s) / len(f1s), 4) if f1s else None,
        "total_test_words": total,
        "skipped_words": len(test_dict) - total,
        "per_word": per_word,
    }


def train():
    lines = load_corpus(corpus_path)
    model_paths = sorted(glob.glob("tokenizers/*.model"))

    report = {}
    for model_path in model_paths:
        name = os.path.basename(model_path).replace(".model", "")
        print(f"eval: {name}")
        sp = spm.SentencePieceProcessor(model_file=model_path)
        report[name] = {
            "corpus_metrics": corpus_metrics(sp, lines),
            "morpheme_metrics": morpheme_metrics(sp, test_tokenizer_dict),
        }

    with open("tokenizer_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print("done: tokenizer_report.json")

train()

eval: bpe_16000
eval: bpe_32000
eval: bpe_50000
eval: bpe_64000
eval: bpe_8000
eval: unigram_16000
eval: unigram_32000
eval: unigram_50000
eval: unigram_64000
eval: unigram_8000
done: tokenizer_report.json


In [8]:
print(511)

511
